# Forced alignment: exact time of every word

For every transcript in `data/transcripts/`, finds where each word is spoken in its `data/raw_audios/` file.
Writes `data/alignments/<name>.json`, one line per word: `{"word", "start", "end"}` (seconds).

**Model:** NVIDIA `stt_uz_fastconformer_hybrid_large_pc` (Uzbek FastConformer, CTC head), with the NeMo Forced Aligner (NFA) functions that ship with NeMo.
The model outputs, every 80 ms, a probability for each token. Viterbi decoding finds the most likely path through *our* text over those frames, which gives each word a start and an end.

**Accuracy notes**
- Times are on an 80 ms grid (the model's frame rate).
- On a test file NFA's word positions had no systematic shift: they were better placed than Whisper's own timestamps, which ran ~0.1 s early.
- CTC models mark each token with a short spike, so a word that is a single token (`odamlar`, `men`) can show a span of just 80 ms, and word ends are often early. When cutting clips, pad a little or cut halfway between words.

Run after `transcribe.ipynb` and `scripts/edit_transcriptions.py`.

In [2]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")  # run ops MPS lacks on CPU (must be set before torch is imported)

import gc, json, logging, re, time, warnings
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
import torchaudio.functional as AF
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
from nemo.utils import logging as nemo_logging
nemo_logging.setLevel(logging.ERROR)
from nemo.collections.asr.models import ASRModel
from nemo.collections.asr.parts.utils.aligner_utils import (
    Segment, Word, add_t_start_end_to_utt_obj, get_batch_variables, viterbi_decoding)

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
audio_dir       = root / "data" / "raw_audios"
transcripts_dir = root / "data" / "transcripts"
output_dir      = root / "data" / "alignments"

MODEL_NAME     = "nvidia/stt_uz_fastconformer_hybrid_large_pc"
DEVICE         = "mps" if torch.backends.mps.is_available() else "cpu"   # runs the acoustic model
VITERBI_DEVICE = "cpu"       # the Viterbi loop is many small steps: faster on CPU
LOCAL_ATTENTION = [64, 64]   # frames of context left/right; makes memory grow linearly with audio length
print("root:", root, "| device:", DEVICE)

/Users/macbook/Documents/IT/projects/tts/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


root: /Users/macbook/Documents/IT/projects/tts | device: mps


## Load the model
Downloaded once (459 MB) into the Hugging Face cache.
- `ctc` decoder: the model is a hybrid (RNN-T + CTC); forced alignment needs the CTC head's per-frame probabilities.
- local attention: with the default full attention, memory grows with the *square* of the audio length, so a 45-minute file would not fit.

In [3]:
model = ASRModel.from_pretrained(MODEL_NAME, map_location="cpu")
model.change_decoding_strategy(decoder_type="ctc", verbose=False)
model.change_attention_model(self_attention_model="rel_pos_local_attn", att_context_size=LOCAL_ATTENTION)
model.change_subsampling_conv_chunking_factor(1)  # 1 = chunk the first conv layers automatically for long audio
model = model.eval().to(DEVICE)

SAMPLE_RATE = model.cfg.preprocessor.sample_rate                                     # 16000
FRAME_SEC = model.cfg.preprocessor.window_stride * model.encoder.subsampling_factor   # 0.08 s per output frame
VOCAB_CHARS = set("".join(model.tokenizer.vocab).replace("▁", ""))
print(type(model).__name__, "| sample rate", SAMPLE_RATE, "| frame", FRAME_SEC, "s")
print("characters the model knows:", "".join(sorted(VOCAB_CHARS)))

EncDecHybridRNNTCTCBPEModel | sample rate 16000 | frame 0.08 s
characters the model knows: ,-.<>?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyzʻ’


## Text normalization (for alignment only)
The text we align must use the model's alphabet, otherwise those characters become `<unk>` tokens that can't be matched to sound:
- **Apostrophes:** Whisper writes `O'zbekiston`, `yo'q`, `ma'no`. The model uses `ʻ` for oʻ / gʻ and `’` for the glottal stop. Fixing this drops unknown tokens from ~6% to ~0.4%.
- **Numbers:** digits aren't in the vocabulary, and the speaker *says* them. `74` becomes `yetmishtoʻrt` and `50%` becomes `ellikfoiz` (spelled out with no spaces, so it stays one word).
- Any other unknown character (`!`, `"`, `&`, stray Cyrillic) is dropped.

The output always keeps Whisper's **original** word. Normalization changes letters *inside* words but never splits or merges words, so word *i* of the normalized text is word *i* of the transcript.

In [4]:
APOSTROPHES = "'‘’ʻʼ`"
ONES = ["nol", "bir", "ikki", "uch", "toʻrt", "besh", "olti", "yetti", "sakkiz", "toʻqqiz"]
TENS = ["", "oʻn", "yigirma", "oʻttiz", "qirq", "ellik", "oltmish", "yetmish", "sakson", "toʻqson"]
SCALES = [(10**9, "milliard"), (10**6, "million"), (1000, "ming")]


def number_to_uz(n):
    """Uzbek cardinal number as words, e.g. 1974 -> 'ming toʻqqiz yuz yetmish toʻrt'."""
    if n < 10:
        return ONES[n]
    if n < 100:
        return " ".join(w for w in (TENS[n // 10], ONES[n % 10] if n % 10 else "") if w)
    if n < 1000:
        head = "yuz" if n // 100 == 1 else f"{ONES[n // 100]} yuz"
        return head + (f" {number_to_uz(n % 100)}" if n % 100 else "")
    for size, name in SCALES:
        if n >= size:
            head = name if (n // size == 1 and size == 1000) else f"{number_to_uz(n // size)} {name}"
            return head + (f" {number_to_uz(n % size)}" if n % size else "")


def spell_digits(m):
    digits = m.group()
    if len(digits) > 12:                                  # phone numbers etc.: digit by digit
        return "".join(ONES[int(d)] for d in digits)
    return number_to_uz(int(digits)).replace(" ", "")     # no spaces: must stay one word


def normalize_word(word):
    w = re.sub(rf"(?<=[oOgG])[{APOSTROPHES}]", "ʻ", word)   # o' g' -> oʻ gʻ
    w = re.sub(rf"(?<![oOgG])[{APOSTROPHES}]", "’", w)       # any other apostrophe -> glottal stop ’
    w = w.replace("%", "foiz")
    w = re.sub(r"\d+", spell_digits, w)
    w = "".join(ch for ch in w if ch in VOCAB_CHARS)        # drop ! " & / stray Cyrillic ...
    return w or "-"                                         # never empty: keeps word i == word i


sample = "O'zbekistonda 74 ta dasturchi bor! 10-15% bo'lsa ham, ma'no bor, g'alaba 1000 marta."
print(" ".join(normalize_word(w) for w in sample.split()))

Oʻzbekistonda yetmishtoʻrt ta dasturchi bor oʻn-oʻnbeshfoiz boʻlsa ham, ma’no bor, gʻalaba ming marta.


## Alignment
`align(audio_path, text)`:
1. loads the audio as 16 kHz mono (the raw files are 44.1 kHz stereo)
2. runs the model to get per-frame token log-probabilities and tokenizes the normalized text (`get_batch_variables`). On MPS, long files fall back to CPU automatically.
3. finds the best path of the text through the frames (`viterbi_decoding`)
4. turns frame indices into seconds for every word (`add_t_start_end_to_utt_obj`)

In [5]:
def load_audio(path):
    audio, sr = sf.read(path, dtype="float32", always_2d=True)
    audio = torch.from_numpy(audio.mean(axis=1))          # stereo -> mono
    if sr != SAMPLE_RATE:
        audio = AF.resample(audio, sr, SAMPLE_RATE)
    return audio.numpy()


def model_outputs(audio, norm_text):
    """Per-frame log-probs + tokenized text. MPS can't run the first conv layer on long audio
    (tens of minutes): then this file is retried on CPU (slower, but works)."""
    def run():
        with torch.inference_mode():
            return get_batch_variables(
                audio=[audio], model=model, gt_text_batch=[norm_text],
                segment_separators=None,            # one segment: we only need words
                output_timestep_duration=FRAME_SEC, verbose=False)
    try:
        return run()
    except (NotImplementedError, RuntimeError) as e:
        if DEVICE == "cpu":
            raise
        print(f"  {DEVICE} failed ({type(e).__name__}: {str(e)[:80]}), retrying on CPU")
        model.to("cpu")
        try:
            return run()
        finally:
            model.to(DEVICE)


def align(audio_path, text):
    original_words = text.split()
    norm_text = " ".join(normalize_word(w) for w in original_words)
    audio = load_audio(audio_path)

    log_probs, y, T, U, utts, frame_sec = model_outputs(audio, norm_text)
    with torch.inference_mode():
        path = viterbi_decoding(log_probs, y, T, U, viterbi_device=VITERBI_DEVICE)
    utt = add_t_start_end_to_utt_obj(utts[0], path[0], frame_sec)

    aligned = [w for seg in utt.segments_and_tokens if isinstance(seg, Segment)
               for w in seg.words_and_tokens if isinstance(w, Word)]
    if len(aligned) != len(original_words):
        raise RuntimeError(f"word count mismatch: {len(aligned)} aligned vs {len(original_words)} in transcript")
    return [{"word": orig, "start": round(w.t_start, 2), "end": round(w.t_end, 2)}
            for orig, w in zip(original_words, aligned)]


def write_alignment(out_path, record):
    """Pretty JSON with one word per line."""
    head = {k: v for k, v in record.items() if k != "words"}
    lines = [f"    {json.dumps(k)}: {json.dumps(v, ensure_ascii=False)}" for k, v in head.items()]
    words = ",\n".join("        " + json.dumps(w, ensure_ascii=False) for w in record["words"])
    lines.append(f'    "words": [\n{words}\n    ]')
    tmp = out_path.with_suffix(".json.tmp")
    tmp.write_text("{\n" + ",\n".join(lines) + "\n}\n", encoding="utf-8")
    os.replace(tmp, out_path)                              # atomic: no half-written files


def align_transcript(json_path):
    t = json.loads(json_path.read_text(encoding="utf-8"))
    audio_path = audio_dir / t["audio"]
    words = align(audio_path, t["text"])
    record = {"audio": t["audio"], "duration": t.get("duration"), "words": words}
    write_alignment(output_dir / json_path.name, record)
    return record

## Smoke test
Aligns the short `abuser.wav` (69 s) and prints the first words. The next cell plays a few words so you can check them by ear.

In [6]:
output_dir.mkdir(parents=True, exist_ok=True)
t0 = time.time()
rec = align_transcript(transcripts_dir / "abuser.json")
print(f"{len(rec['words'])} words in {time.time() - t0:.1f}s")
for w in rec["words"][:25]:
    print(f"{w['start']:7.2f} → {w['end']:7.2f}  {w['word']}")

[NeMo W 2026-11-24 16:34:55 ctc_greedy_decoding:278] CTC decoding strategy 'greedy' is slower than 'greedy_batch', which implements the same exact interface. Consider changing your strategy to 'greedy_batch' for a free performance improvement.


186 words in 0.3s
   0.00 →    0.64  O'zbekistonda
   0.72 →    1.12  Naviyni
   1.20 →    1.52  yutgan
   1.76 →    1.84  odamlar
   1.92 →    2.16  borligini
   2.24 →    3.04  bilarmidinglar?
   3.12 →    3.20  Bir
   3.36 →    3.92  oltidan.
   3.92 →    4.24  Yo'q,
   4.24 →    4.56  yo'q,
   4.56 →    4.64  men
   4.64 →    4.88  xabarim
   4.88 →    5.20  yo'q.
   5.20 →    5.28  Bir
   5.44 →    6.16  oltida,
   6.40 →    6.48  hozir
   6.80 →    6.88  ham
   6.88 →    7.44  KSKda
   7.44 →    7.52  ham
   7.52 →    8.00  o'ynaydi,
   8.00 →    8.32  Sekka
   8.48 →    8.56  degan
   8.56 →    8.64  odam
   8.80 →    9.04  bor.
   9.12 →    9.20  Bir


In [7]:
from IPython.display import Audio, display

def play(record, i, j=None, pad=0.15):
    """Play words i..j (inclusive) of an alignment record."""
    j = i if j is None else j
    start, end = record["words"][i]["start"] - pad, record["words"][j]["end"] + pad
    audio = load_audio(audio_dir / record["audio"])
    print(" ".join(w["word"] for w in record["words"][i:j + 1]), f"[{start:.2f}–{end:.2f}s]")
    display(Audio(audio[int(max(start, 0) * SAMPLE_RATE):int(end * SAMPLE_RATE)], rate=SAMPLE_RATE))

play(rec, 0, 5)
play(rec, 100)

O'zbekistonda Naviyni yutgan odamlar borligini bilarmidinglar? [-0.15–3.19s]


Navi [39.85–40.39s]


## Full run
Aligns every transcript that has no alignment file yet. Safe to interrupt and re-run.

In [8]:
output_dir.mkdir(parents=True, exist_ok=True)
todo = [p for p in sorted(transcripts_dir.glob("*.json")) if not (output_dir / p.name).exists()]
print(len(todo), "to align")

failed = []
for json_path in tqdm(todo):
    t0 = time.time()
    try:    
        rec = align_transcript(json_path)
        print(f"ok   {json_path.stem[:60]}  {len(rec['words'])} words, {time.time() - t0:.0f}s")
    except Exception as e:
        failed.append(json_path.name)
        print(f"FAIL {json_path.stem[:60]}: {e}")
    gc.collect()
    if DEVICE == "mps":
        torch.mps.empty_cache()
print("failed:", failed)

30 to align


  0%|          | 0/30 [00:00<?, ?it/s]

  mps failed (NotImplementedError: convolution_overrideable not implemented. You are likely triggering this with te), retrying on CPU


  3%|▎         | 1/30 [02:19<1:07:32, 139.73s/it]

ok   18-35 yosh bo'lsangiz, shu 8ta darsni eshitishingiz kerak  18471 words, 140s


  7%|▋         | 2/30 [02:26<28:37, 61.35s/it]   

... (51 outputs left out)

 93%|█████████▎| 28/30 [43:48<03:17, 98.75s/it] 

ok   Xonadagi eng aqlli bo'lish yo'li - Iqtibos podcast #38  5006 words, 8s
ok   [PODKAST 013] Odam bo'laylik podkasti Davron Fayziyev bilan  7058 words, 29s


100%|██████████| 30/30 [44:21<00:00, 88.71s/it]

ok   hayotingiz isrof bo'lishiga sabab (yo'q, telefon emas)  2016 words, 2s
failed: []
